# Datathon - Fase 5
Case Passos Mágicos

# Modelo Preditivo de Risco

## Introdução

### 9. Previsão de risco com Machine Learning
Quais padrões nos indicadores permitem identificar alunos em risco antes
de queda no desempenho ou aumento da defasagem? Construa um modelo
preditivo que mostre uma probabilidade do aluno ou aluna entrar em risco de
defasagem.

### Objetivo
Construir um modelo preditivo capaz de identificar alunos com risco de defasagem,
com base em indicadores acadêmicos, comportamentais e psicopedagógicos.

## Preparação dos Dados

### Bibliotecas

In [4]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Métricas de avaliação
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

### Importação da Base

In [5]:
# Importar o arquivo do github
url = "https://raw.githubusercontent.com/LaisST/FIAP_Datathon_Fase_5/refs/heads/main/Bases/base_explorada.csv"

df_explorada = pd.read_csv(url)

In [6]:
#
df_explorada.head(5)

,RA,Fase,Turma,Nome,Data de Nasc,Idade,Gênero,Ano ingresso,Instituição de ensino,Pedra,...,Ano_Referencia,Instituicao_cat,Fase_num,Indicado_flag,Atingiu_PV_flag,nivel_risco,faixa_IEG,faixa_IAA,faixa_IPS,faixa_INDE
0,RA-1,7,A,Aluno-1,2003-01-01,19.0,Feminino,2016,Escola Pública,Quartzo,...,2022,Pública,7,1.0,0.0,Risco moderado,Baixo,Média,Baixo,Baixo
1,RA-2,7,A,Aluno-2,2005-01-01,17.0,Feminino,2017,Rede Decisão,Ametista,...,2022,Outros,7,0.0,0.0,Sem risco,Baixo,Média,Médio,Médio
2,RA-3,7,A,Aluno-3,2005-01-01,17.0,Feminino,2016,Rede Decisão,Ágata,...,2022,Outros,7,0.0,0.0,Sem risco,Médio,Baixa,Baixo,Baixo
3,RA-4,7,A,Aluno-4,2005-01-01,17.0,Masculino,2017,Rede Decisão,Quartzo,...,2022,Outros,7,0.0,0.0,Sem risco,Baixo,Média,Baixo,Baixo
4,RA-5,7,A,Aluno-5,2005-01-01,17.0,Feminino,2016,Rede Decisão,Ametista,...,2022,Outros,7,0.0,0.0,Sem risco,Médio,Baixa,Baixo,Médio


In [7]:
# Tipos de Dados e checar os nomes das colunas
df_explorada.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3030 entries, 0 to 3029
Data columns (total 37 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   RA                     3030 non-null   object 
 1   Fase                   3030 non-null   object 
 2   Turma                  3030 non-null   object 
 3   Nome                   3030 non-null   object 
 4   Data de Nasc           860 non-null    object 
 5   Idade                  2631 non-null   float64
 6   Gênero                 3030 non-null   object 
 7   Ano ingresso           3030 non-null   int64  
 8   Instituição de ensino  3029 non-null   object 
 9   Pedra                  2845 non-null   object 
 10  INDE                   2845 non-null   float64
 11  Nº Av                  2954 non-null   float64
 12  IAA                    2865 non-null   float64
 13  IEG                    2954 non-null   float64
 14  IPS                    2859 non-null   float64
 15  Rec 

### Seleção das variáveis

In [8]:
features = ['IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV']

### Criação da Target

In [9]:
# Função para criação da target com base na coluna nivel_risco
def definir_target(x):
    if x == 'Sem risco':
        return 0
    else:
        return 1

df_explorada['target_risco'] = df_explorada['nivel_risco'].apply(definir_target)

### Separação das feratures (X) e Target (y)

In [10]:
X = df_explorada[features]
y = df_explorada['target_risco']

### Tratamento de valores nulos
Substituir os valores nulos pela mediana da coluna.

In [11]:
X = X.fillna(X.median())

### Divisão da base em Treino e Teste
Treino (70%) o modelo aprende

Teste (30%) o modelo é avaliado

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

### Treinamento do modelo em Random Forest

In [13]:
modelo = RandomForestClassifier(random_state=42)
modelo.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

### Previsão e Avaliação do Modelo Random Forest



In [14]:
y_pred = modelo.predict(X_test)
y_prob = modelo.predict_proba(X_test)[:,1]

In [15]:
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[183 220]
 [136 370]]
              precision    recall  f1-score   support

           0       0.57      0.45      0.51       403
           1       0.63      0.73      0.68       506

    accuracy                           0.61       909
   macro avg       0.60      0.59      0.59       909
weighted avg       0.60      0.61      0.60       909



### Variaveis mais usadas no Modelo

In [16]:
importancias = pd.DataFrame({
    'Variavel': features,
    'Importancia': modelo.feature_importances_
}).sort_values(by='Importancia', ascending=False)
importancias

,Variavel,Importancia
5,IPV,0.219794
1,IEG,0.210774
0,IDA,0.198558
2,IAA,0.139340
3,IPS,0.122923
4,IPP,0.108611


O modelo alcançou 61% de acurácia e identificou 73% dos alunos em risco. Os principais fatores associados ao risco são o ponto de virada, o engajamento e o desempenho. Isso indica que aspectos comportamentais são fundamentais para antecipar a defasagem.

#### Salvar os dados em CSV

In [22]:
# Criar DF com os resultados Metricas
metricas = pd.DataFrame({
    'Metrica': ['Accuracy', 'Precision', 'Recall', 'F1-score'],
    'Valor': [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred)
    ]
})

# Salvar em csv
metricas.to_csv('metricas_modelo.csv', index=False)

In [23]:
# Criar DF com a importancia de cada indicador
importancia = pd.DataFrame({
    'Variavel': features,
    'Importancia': modelo.feature_importances_
}).sort_values(by='Importancia', ascending=False)

#Salvar em csv
importancia.to_csv('importancia_variaveis.csv', index=False)